# ULR-ABL Comparative Experiment

This notebook runs a comparative study between:
1. **Baseline**: ULR semantic segmentation WITHOUT Active Boundary Loss
2. **ABL Model**: ULR semantic segmentation WITH Active Boundary Loss

## Data Safety

✅ **Protected from overwrites**: Each experiment uses a separate checkpoint directory:
- Baseline: `{EXPERIMENT_RESULTS_DIR}/checkpoints/baseline/`
- ABL: `{EXPERIMENT_RESULTS_DIR}/checkpoints/with_abl/`

⚠️ **Warning system**: If you re-run an experiment with the same date, the code will warn before overwriting files.

💡 **Recommendation**: Change `EXPERIMENT_RESULTS_DIR` date suffix or add a run identifier to preserve multiple runs.

In [1]:
!nvidia-smi

Mon Feb  2 13:05:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ============================================================================
# CONFIGURATION: Dataset and experiment paths
# ============================================================================
from datetime import datetime

# Experiment results directory (Google Drive)
EXPERIMENT_RESULTS_DIR = f'/content/drive/MyDrive/ULR_experiment{datetime.now().strftime("%m-%d")}'

# Source dataset paths
SOURCE_RGB = '/content/drive/MyDrive/customULR/custom_rgb'
SOURCE_LABEL = '/content/drive/MyDrive/customULR/custom_label'

TRAIN_EPOCHS = 20
PRETRAIN_EPOCHS = 7
TRAIN_RATIO = 0.7

# Train/test split paths (local)
TRAIN_RGB = '/content/data/train/rgb'
TRAIN_LABEL = '/content/data/train/label'
TEST_RGB = '/content/data/test/rgb'
TEST_LABEL = '/content/data/test/label'

# Batch inference paths (separate from evaluation test set)
BATCH_INFERENCE_RGB = '/content/drive/MyDrive/customULR/inference_samples/rgb'
BATCH_INFERENCE_LABEL = '/content/drive/MyDrive/customULR/inference_samples/label'

# Random seed for reproducibility
SEED = 42

In [3]:
# Clone the ULR-ABL experiment repository
!git clone https://github.com/IgnasJo/ULR-experiment.git
%cd ULR-experiment
!ls

Cloning into 'ULR-experiment'...
remote: Enumerating objects: 303, done.
remote: Counting objects: 100% (303/303), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 303 (delta 144), reused 281 (delta 122), pack-reused 0 (from 0)
Receiving objects: 100% (303/303), 15.32 MiB | 11.43 MiB/s, done.
Resolving deltas: 100% (144/144), done.
/content/ULR-experiment
abl			esrgan.py	  modeling	      training
batch_inference.py	evaluation.py	  PIPELINE_README.md  training.py
check_image_classes.py	full_pipeline.py  pretraining.py      utils2
checkpoints		GEMINI.md	  README.md
config.py		inference.py	  requirements.txt


In [4]:
# Mount Google Drive for dataset and checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

# Create directories for checkpoints
import os
os.makedirs(f'{EXPERIMENT_RESULTS_DIR}/checkpoints', exist_ok=True)

KeyboardInterrupt: 

In [ ]:
# Create 70/30 train/test split
from training.utils import create_train_test_split

create_train_test_split(
    source_rgb=SOURCE_RGB,
    source_label=SOURCE_LABEL,
    train_rgb=TRAIN_RGB,
    train_label=TRAIN_LABEL,
    test_rgb=TEST_RGB,
    test_label=TEST_LABEL,
    train_ratio=TRAIN_RATIO,
    seed=SEED
)

In [ ]:
# ============================================================================
# Helper: Configure experiment via environment variables (works with subprocesses)
# ============================================================================
import os

def configure_experiment(checkpoint_dir: str, use_abl: bool = False, num_epochs: int = TRAIN_EPOCHS):
    """
    Configure the experiment via environment variables.
    These are picked up by config.py in subprocess calls.
    
    Args:
        checkpoint_dir: Full path to checkpoint directory for this experiment
        use_abl: Whether to enable Active Boundary Loss
        num_epochs: Number of training epochs
    """
    # Set environment variables (inherited by subprocesses)
    os.environ['ULR_CHECKPOINT_DIR'] = checkpoint_dir
    os.environ['ULR_TRAIN_RGB'] = TRAIN_RGB
    os.environ['ULR_TRAIN_LABEL'] = TRAIN_LABEL
    os.environ['ULR_TEST_RGB'] = TEST_RGB
    os.environ['ULR_TEST_LABEL'] = TEST_LABEL
    os.environ['ULR_TRAIN_EPOCHS'] = str(num_epochs)
    os.environ['ULR_PRETRAIN_EPOCHS'] = str(PRETRAIN_EPOCHS)
    os.environ['ULR_USE_ABL'] = str(use_abl)
    os.environ['ULR_BATCH_SIZE'] = '2'  # Optimized for T4 GPU (15GB VRAM)
    os.environ['ULR_ALPHA'] = '0.4'
    os.environ['ULR_LAMBDA_3'] = '0.0025'
    os.environ['ULR_LAMBDA_ABL'] = '0.02'
    
    abl_status = "WITH" if use_abl else "WITHOUT"
    print(f"Configured experiment {abl_status} ABL")
    print(f"  Checkpoints: {checkpoint_dir}")
    print(f"  Epochs: {num_epochs}")
    print(f"  Batch size: 2")

In [ ]:
# ============================================================================
# EXPERIMENT 1: Training WITHOUT Active Boundary Loss (Baseline)
# ============================================================================
BASELINE_DIR = f'{EXPERIMENT_RESULTS_DIR}/checkpoints/baseline'
configure_experiment(checkpoint_dir=BASELINE_DIR, use_abl=False)

In [ ]:
# Run Experiment 1: Training WITHOUT ABL (Baseline)
print("=" * 60)
print("EXPERIMENT 1: Training WITHOUT Active Boundary Loss")
print("=" * 60)
!python full_pipeline.py

In [ ]:
# ============================================================================
# EXPERIMENT 2: Training WITH Active Boundary Loss
# ============================================================================
ABL_DIR = f'{EXPERIMENT_RESULTS_DIR}/checkpoints/with_abl'
configure_experiment(checkpoint_dir=ABL_DIR, use_abl=True)

In [ ]:
# Run Experiment 2: Training WITH ABL (reusing pretrained weights from Exp 1)
print("=" * 60)
print("EXPERIMENT 2: Training WITH Active Boundary Loss")
print("=" * 60)
# Use pretrained weights from baseline experiment
!python full_pipeline.py --skip-pretrain --pretrained-gen {BASELINE_DIR}/pretrained_generator.pth --pretrained-disc {BASELINE_DIR}/pretrained_discriminator.pth

In [ ]:
# ============================================================================
# EVALUATION: Compare both models on test set
# ============================================================================
print("=" * 60)
print("EVALUATION: Comparing Baseline vs ABL models")
print("=" * 60)

# Evaluate baseline model
print("\n--- Baseline (No ABL) Results ---")
configure_experiment(checkpoint_dir=BASELINE_DIR, use_abl=False)
!python full_pipeline.py --eval-only --eval-output {EXPERIMENT_RESULTS_DIR}/evaluation_baseline

# Evaluate ABL model  
print("\n--- With ABL Results ---")
configure_experiment(checkpoint_dir=ABL_DIR, use_abl=False)  # ABL not needed for eval
!python full_pipeline.py --eval-only --eval-output {EXPERIMENT_RESULTS_DIR}/evaluation_with_abl

In [ ]:
# ============================================================================
# BATCH INFERENCE: Generate visual outputs for both models
# ============================================================================
print("=" * 60)
print("BATCH INFERENCE: Generating outputs for Baseline vs ABL models")
print("=" * 60)

# Set batch inference paths
import os
os.environ['ULR_BATCH_INFERENCE_RGB'] = BATCH_INFERENCE_RGB
os.environ['ULR_BATCH_INFERENCE_LABEL'] = BATCH_INFERENCE_LABEL

# Batch inference for baseline model
print("\n--- Baseline (No ABL) Inference ---")
configure_experiment(checkpoint_dir=BASELINE_DIR, use_abl=False)
!python full_pipeline.py --batch-inference --eval-output {EXPERIMENT_RESULTS_DIR}/inference_baseline

# Batch inference for ABL model  
print("\n--- With ABL Inference ---")
configure_experiment(checkpoint_dir=ABL_DIR, use_abl=False)  # ABL not needed for inference
!python full_pipeline.py --batch-inference --eval-output {EXPERIMENT_RESULTS_DIR}/inference_with_abl